# Single-Organoid Lineage Graph

Load one preprocessed organoid graph, derive collaborator-style mutually exclusive lineage intensities using the same helper as `scripts/export_marker_intensities_to_mesh.py`, and plot the graph with lineage names in the legend.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Resolve the repository root whether the notebook is launched from the repo root
# or directly from the notebooks/ directory.
CWD = Path.cwd().resolve()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Pick one organoid graph. Set LABEL_UID=None to use the first graph in the index.
DATASET = "20250929"
TIMEPOINT = "day4p5"
LABEL_UID = "day4p5_A06_61"

DATA_ROOT = REPO_ROOT.parent / "NicoleData" / DATASET
GRAPHS_DIR = DATA_ROOT / "graphs_preprocessed"
CELL_CONFIG_PATH = DATA_ROOT / "cell_table_config.json"

# Graphs loaded from graphs_preprocessed should use processed marker intensities.
MARKER_SOURCE_FIELD = "markers_int"

LINEAGE_COLORS = {
    "STEM": "#1b9e77",
    "EEPROG": "#7570b3",
    "GOBLET": "#e7298a",
    "ABS": "#66a61e",
    "EE": "#e6ab02",
    "SECPROG": "#a6761d",
    "EC": "#d95f02",
    "PANETH": "#e41a1c",
    "TA": "#377eb8",
}

# Plot settings.
NODE_SIZE = 5
PROJECTED_CELL_SIZE = 4
EDGE_WIDTH = 0.6
FIG_SIZE = (850, 650)
BASELINE_COLOR = "lightgray"
BASELINE_LABEL = "unassigned"
MESH_ALPHA = 0.18
CELL_PATCH_ALPHA = 0.95
UNOWNED_SURFACE_COLOR = "#eeeeee"
SHOW_PROJECTED_CELL_POINTS = False

print("REPO_ROOT =", REPO_ROOT)
print("DATA_ROOT =", DATA_ROOT)


REPO_ROOT = /home/fmoller/Projects/LearningOrganoids/OrganoGraph
DATA_ROOT = /home/fmoller/Projects/LearningOrganoids/NicoleData/20250929


In [2]:
import copy

import networkx as nx
import plotly.graph_objects as go

from organograph.graph.access import graph_get
from organograph.graph.io import load_cell_graph
from organograph.io_utils.dataset_config import load_cell_table_config
from organograph.mesh.OrganoidMesh import OrganoidMesh
from organograph.plotting.graphs import plot_graph_by_markers
from organograph.plotting.meshes import plot_organoid_mesh

from scripts.export_marker_intensities_to_mesh import (
    remap_owner_to_node,
    apply_collaborator_exclusivity_to_intensities,
)


In [3]:
index_path = GRAPHS_DIR / TIMEPOINT / "index.csv"
if not index_path.exists():
    raise FileNotFoundError(f"Graph index not found: {index_path}")

index_df = pd.read_csv(index_path)
if LABEL_UID is None:
    graph_row = index_df.iloc[0]
else:
    matches = index_df[index_df["label_uid"].astype(str) == str(LABEL_UID)]
    if matches.empty:
        available = index_df["label_uid"].astype(str).head(10).tolist()
        raise ValueError(
            f"LABEL_UID={LABEL_UID!r} was not found in {index_path}. "
            f"First available labels: {available}"
        )
    graph_row = matches.iloc[0]

graph_path = Path(str(graph_row["graph_path"])).expanduser()
if not graph_path.exists():
    raise FileNotFoundError(f"Graph file not found: {graph_path}")

G = load_cell_graph(graph_path)
graph_marker_names = [str(x) for x in G.graph.get("marker_names", [])]

cell_cfg = load_cell_table_config(CELL_CONFIG_PATH)
marker_cols = list(cell_cfg["marker_cols"])
config_marker_names = [str(x) for x in cell_cfg.get("marker_names", marker_cols)]

if config_marker_names != graph_marker_names:
    raise ValueError(
        "cell_table_config marker_names do not match the graph marker_names.\n"
        f"Config: {config_marker_names}\n"
        f"Graph:  {graph_marker_names}"
    )
if len(marker_cols) != len(graph_marker_names):
    raise ValueError(
        f"cell_table_config has {len(marker_cols)} marker columns but the graph has "
        f"{len(graph_marker_names)} markers."
    )

print("label_uid =", graph_row["label_uid"])
print("graph_path =", graph_path)
print("n_cells =", G.number_of_nodes())
print("n_edges =", G.number_of_edges())
print("graph markers =", graph_marker_names)


label_uid = day4p5_A06_61
graph_path = /home/fmoller/Projects/LearningOrganoids/OrganoGraph/../NicoleData/20250929/graphs_preprocessed/day4p5/day4p5_A06_61.gpickle
n_cells = 930
n_edges = 2784
graph markers = ['LGR5', 'Chroma', 'Cyclin D', 'Mucin 2', 'AldoB', 'Glucagon', 'Cyclin A', 'Agr2', 'Serotonin', 'Lysozyme']


## Lysozyme Coexpression and Cluster Diagnostics

Before applying the collaborator exclusivity rules, inspect Lysozyme-positive cells in the selected marker source. For each Lysozyme-positive cell, the table reports coexpressed markers and the size of the connected Lysozyme-positive graph component containing that cell.


In [ ]:
def graph_has_node_field(graph, field):
    return graph.number_of_nodes() > 0 and field in graph.nodes[0]


def resolve_marker_index(marker_names, candidates):
    norm_to_idx = {str(name).strip().lower(): i for i, name in enumerate(marker_names)}
    for candidate in candidates:
        idx = norm_to_idx.get(str(candidate).strip().lower())
        if idx is not None:
            return idx
    return None


marker_source_field = MARKER_SOURCE_FIELD
markers_int = graph_get(G, marker_source_field, dtype=float)
markers_pos = markers_int > 0

lysozyme_idx = resolve_marker_index(graph_marker_names, ["Lysozyme", "LYZ"])
if lysozyme_idx is None:
    raise ValueError(f"No Lysozyme marker found. Available markers: {graph_marker_names}")

lysozyme_nodes = np.flatnonzero(markers_pos[:, lysozyme_idx]).astype(int)
lysozyme_node_set = set(lysozyme_nodes.tolist())
lysozyme_component_size = np.zeros(G.number_of_nodes(), dtype=np.int64)
lysozyme_component_id = np.full(G.number_of_nodes(), -1, dtype=np.int64)

for component_id, component in enumerate(nx.connected_components(G.subgraph(lysozyme_nodes.tolist())), start=1):
    component = np.asarray(sorted(component), dtype=np.int64)
    lysozyme_component_size[component] = int(component.size)
    lysozyme_component_id[component] = int(component_id)

rows = []
for node in lysozyme_nodes:
    positive_marker_idx = np.flatnonzero(markers_pos[node]).astype(int)
    coexpressed_idx = positive_marker_idx[positive_marker_idx != lysozyme_idx]
    coexpressed_markers = [graph_marker_names[j] for j in coexpressed_idx]
    rows.append({
        "node": int(node),
        "cell_index": int(G.nodes[int(node)].get("cell_index", node)),
        "lysozyme_intensity": float(markers_int[node, lysozyme_idx]),
        "lysozyme_cluster_id": int(lysozyme_component_id[node]),
        "lysozyme_cluster_size": int(lysozyme_component_size[node]),
        "n_coexpressed_markers": int(len(coexpressed_markers)),
        "coexpressed_markers": ", ".join(coexpressed_markers) if coexpressed_markers else "none",
    })

lysozyme_cell_table = pd.DataFrame(rows).sort_values(
    ["lysozyme_cluster_size", "lysozyme_cluster_id", "node"],
    ascending=[False, True, True],
).reset_index(drop=True)

coexpression_summary = []
for j, marker in enumerate(graph_marker_names):
    if j == lysozyme_idx:
        continue
    mask = markers_pos[lysozyme_nodes, j] if lysozyme_nodes.size else np.array([], dtype=bool)
    coexpression_summary.append({
        "marker": marker,
        "n_lysozyme_cells_coexpressing": int(mask.sum()),
        "fraction_of_lysozyme_cells": float(mask.mean()) if mask.size else np.nan,
    })
coexpression_summary = pd.DataFrame(coexpression_summary).sort_values(
    ["n_lysozyme_cells_coexpressing", "marker"],
    ascending=[False, True],
).reset_index(drop=True)

cluster_summary = (
    lysozyme_cell_table
    .groupby(["lysozyme_cluster_id", "lysozyme_cluster_size"], as_index=False)
    .agg(
        n_cells=("node", "size"),
        nodes=("node", lambda x: ", ".join(map(str, x))),
        coexpressed_markers_in_cluster=(
            "coexpressed_markers",
            lambda values: ", ".join(
                sorted({m.strip() for value in values for m in str(value).split(",") if m.strip() and m.strip() != "none"})
            ) or "none",
        ),
    )
    .sort_values(["lysozyme_cluster_size", "lysozyme_cluster_id"], ascending=[False, True])
    .reset_index(drop=True)
    if not lysozyme_cell_table.empty
    else pd.DataFrame(columns=["lysozyme_cluster_id", "lysozyme_cluster_size", "n_cells", "nodes", "coexpressed_markers_in_cluster"])
)

print("marker_source_field =", marker_source_field)
print(f"Lysozyme-positive cells: {len(lysozyme_nodes)} / {G.number_of_nodes()}")
print(f"Lysozyme-positive clusters: {cluster_summary.shape[0]}")
display(coexpression_summary)
display(cluster_summary)
display(lysozyme_cell_table)


In [ ]:
if "marker_source_field" not in globals():
    marker_source_field = MARKER_SOURCE_FIELD
if "markers_int" not in globals():
    markers_int = graph_get(G, marker_source_field, dtype=float)
lineage_int, lineage_bin, lineage_names, _ = apply_collaborator_exclusivity_to_intensities(
    markers_int,
    graph_marker_names,
    marker_cols,
)
lineage_bin = np.asarray(lineage_bin, dtype=np.int8)

# Build a temporary plotting graph whose marker matrix is the derived lineage
# matrix. The original loaded graph G is left unchanged.
G_lineage = copy.deepcopy(G)
for node in range(G_lineage.number_of_nodes()):
    G_lineage.nodes[node]["markers_int"] = lineage_int[node].astype(float).tolist()
    G_lineage.nodes[node]["markers_bin"] = lineage_bin[node].astype(np.int8).tolist()
G_lineage.graph["marker_names"] = list(lineage_names)
G_lineage.graph["lineage_marker_source_field"] = marker_source_field

lineage_counts = pd.DataFrame({
    "lineage": lineage_names,
    "n_positive": lineage_bin.sum(axis=0).astype(int),
})
lineage_counts["fraction"] = lineage_counts["n_positive"] / max(G_lineage.number_of_nodes(), 1)
lineage_counts.loc[len(lineage_counts)] = {
    "lineage": BASELINE_LABEL,
    "n_positive": int(np.sum(lineage_bin.sum(axis=1) == 0)),
    "fraction": float(np.mean(lineage_bin.sum(axis=1) == 0)),
}

print("marker_source_field =", marker_source_field)
display(lineage_counts)


In [ ]:
marker_map = [
    {
        "marker": lineage,
        "color": LINEAGE_COLORS.get(lineage, "#333333"),
        "name": lineage,
    }
    for lineage in lineage_names
]

fig = plot_graph_by_markers(
    G_lineage,
    marker_map,
    backend="plotly",
    baseline_color=BASELINE_COLOR,
    node_size=NODE_SIZE,
    edge_width=EDGE_WIDTH,
    priority="first",
    add_legend=True,
    legend_baseline_name=BASELINE_LABEL,
    fig_size=FIG_SIZE,
    edges="thin",
)
fig.update_layout(
    title=f"{graph_row['label_uid']} - collaborator exclusivity lineages",
    legend_title_text="Lineage",
)
fig


## Mesh With Projected Lineage Cell Patches

Load the mesh paired with the selected graph and shade the surface patch assigned to each projected cell, after applying the mutually exclusive lineage labels.


In [ ]:
mesh_path = Path(str(graph_row["mesh_path"])).expanduser()
if not mesh_path.exists():
    raise FileNotFoundError(f"Mesh file not found: {mesh_path}")

vertex_owner_value = graph_row.get("vertex_owner_path", None)
if vertex_owner_value is not None and pd.notna(vertex_owner_value) and str(vertex_owner_value):
    vertex_owner_path = Path(str(vertex_owner_value)).expanduser()
else:
    vertex_owner_path = graph_path.with_suffix(".vertex_owner.npz")
if not vertex_owner_path.exists():
    vertex_owner_path = graph_path.with_suffix(".vertex_owner.npz")
if not vertex_owner_path.exists():
    raise FileNotFoundError(f"Vertex-owner sidecar not found: {vertex_owner_path}")

mesh = OrganoidMesh(str(mesh_path))
proj_vertex = graph_get(G, "proj_vertex", dtype=np.int64)
if proj_vertex.shape[0] != G.number_of_nodes():
    raise ValueError(
        f"proj_vertex has {proj_vertex.shape[0]} entries but graph has {G.number_of_nodes()} nodes"
    )

sidecar = np.load(vertex_owner_path, allow_pickle=True)
vertex_owner_table = np.asarray(sidecar["vertex_owner"], dtype=np.int64)
if vertex_owner_table.shape[0] != mesh.v.shape[0]:
    raise ValueError(
        f"vertex_owner has {vertex_owner_table.shape[0]} entries but mesh has "
        f"{mesh.v.shape[0]} vertices"
    )
vertex_owner_node = remap_owner_to_node(vertex_owner_table, G)

lineage_label_by_node = np.full(G.number_of_nodes(), BASELINE_LABEL, dtype=object)
lineage_color_by_node = np.full(G.number_of_nodes(), BASELINE_COLOR, dtype=object)
assigned = np.zeros(G.number_of_nodes(), dtype=bool)
for j, lineage in enumerate(lineage_names):
    mask = (lineage_bin[:, j] > 0) & ~assigned
    lineage_label_by_node[mask] = lineage
    lineage_color_by_node[mask] = LINEAGE_COLORS.get(lineage, "#333333")
    assigned[mask] = True

vertex_colors = np.full(mesh.v.shape[0], UNOWNED_SURFACE_COLOR, dtype=object)
valid_owner = (vertex_owner_node >= 0) & (vertex_owner_node < G.number_of_nodes())
vertex_colors[valid_owner] = lineage_color_by_node[vertex_owner_node[valid_owner]]

# Color each triangle by the majority owner of its three vertices. This keeps
# the cell surface patches discrete instead of interpolating colors per vertex.
faces = np.asarray(mesh.f, dtype=np.int64)
face_owner_node = np.full(faces.shape[0], -1, dtype=np.int64)
for face_i, tri in enumerate(faces):
    owners = vertex_owner_node[tri]
    owners = owners[(owners >= 0) & (owners < G.number_of_nodes())]
    if owners.size == 0:
        continue
    unique, counts = np.unique(owners, return_counts=True)
    face_owner_node[face_i] = int(unique[np.argmax(counts)])

face_colors = np.full(faces.shape[0], UNOWNED_SURFACE_COLOR, dtype=object)
valid_face_owner = face_owner_node >= 0
face_colors[valid_face_owner] = lineage_color_by_node[face_owner_node[valid_face_owner]]

vertices = np.asarray(mesh.v, dtype=float)
fig_mesh = go.Figure(
    data=[
        go.Mesh3d(
            x=vertices[:, 0],
            y=vertices[:, 1],
            z=vertices[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            facecolor=face_colors.tolist(),
            opacity=float(CELL_PATCH_ALPHA),
            flatshading=True,
            hoverinfo="skip",
            name="cell surface patches",
            showlegend=False,
        )
    ]
)

if SHOW_PROJECTED_CELL_POINTS:
    valid_projection = (
        np.isfinite(proj_vertex)
        & (proj_vertex >= 0)
        & (proj_vertex < mesh.v.shape[0])
    )
    projected_xyz = vertices[proj_vertex[valid_projection]]
    projected_labels = lineage_label_by_node[valid_projection]
    for lineage in list(lineage_names) + [BASELINE_LABEL]:
        mask = projected_labels == lineage
        if not np.any(mask):
            continue
        color = BASELINE_COLOR if lineage == BASELINE_LABEL else LINEAGE_COLORS.get(lineage, "#333333")
        pts = projected_xyz[mask]
        fig_mesh.add_trace(
            go.Scatter3d(
                x=pts[:, 0],
                y=pts[:, 1],
                z=pts[:, 2],
                mode="markers",
                marker=dict(size=PROJECTED_CELL_SIZE, color=color, line=dict(width=0)),
                name=f"{lineage} projected center",
                showlegend=False,
                hoverinfo="skip",
            )
        )

# Legend swatches for the lineage-colored surface patches.
for lineage in list(lineage_names) + [BASELINE_LABEL]:
    color = BASELINE_COLOR if lineage == BASELINE_LABEL else LINEAGE_COLORS.get(lineage, "#333333")
    if lineage == BASELINE_LABEL:
        owned_labels = lineage_label_by_node[vertex_owner_node[valid_owner]]
        present = bool(np.any(owned_labels == BASELINE_LABEL))
    else:
        lineage_idx = list(lineage_names).index(lineage)
        present = bool(np.any(lineage_bin[:, lineage_idx] > 0))
    if not present:
        continue
    fig_mesh.add_trace(
        go.Scatter3d(
            x=[None],
            y=[None],
            z=[None],
            mode="markers",
            marker=dict(size=8, color=color),
            name=str(lineage),
            showlegend=True,
        )
    )

fig_mesh.update_layout(
    title=f"{graph_row['label_uid']} - cell surface patches after collaborator exclusivity",
    legend_title_text="Lineage",
    width=int(FIG_SIZE[0]),
    height=int(FIG_SIZE[1]),
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        bgcolor="rgba(0,0,0,0)",
        aspectmode="data",
    ),
    margin=dict(l=0, r=0, t=45, b=0),
)

print("mesh_path =", mesh_path)
print("vertex_owner_path =", vertex_owner_path)
print(f"owned surface vertices: {int(valid_owner.sum())} / {mesh.v.shape[0]}")
fig_mesh
